# TRAINING MODEL LSTM (DEFENSIVE MODEL): PREDIKSI HARGA KRIPTO
**Skripsi: Analisis Sentimen Figur Publik & Prediksi Pasar (2021-2025)**

Notebook ini menggunakan arsitektur LSTM yang diperkuat dengan **L2 Regularization** dan **Shuffle** untuk mencegah overfitting dan bias koin. Hubungkan ke **Google Colab GPU** untuk performa maksimal.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.utils import shuffle
from google.colab import drive

gpu_devices = tf.config.list_physical_devices('GPU')
print("--- MEMULAI PROSES PREDIKSI KRIPTO ---")
print(f"GPU Tersedia: {len(gpu_devices) > 0} ({gpu_devices if gpu_devices else 'CPU'})")

### 1. Koneksi Google Drive & Load Dataset
Pastikan folder `pelatihan lstm` berisi file `.npy` yang sudah di-rebuild.

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

path_drive = '/content/drive/MyDrive/pelatihan lstm/'

try:
    print(f"\n1. Memuat dataset dari: {path_drive}...")
    X_train = np.load(path_drive + 'X_train_3D.npy').astype('float32')
    X_test = np.load(path_drive + 'X_test_3D.npy').astype('float32')
    y_train = np.load(path_drive + 'y_train_3D.npy').astype('int8')
    y_test = np.load(path_drive + 'y_test_3D.npy').astype('int8')
    
    print(f"✅ Data Berhasil Dimuat!")
    print(f"   X_train: {X_train.shape}")
    print(f"   X_test:  {X_test.shape}")

    # 1.5 SHUFFLE DATA (ANTI-BIAS KOIN)
    print("\n[INFO] Mengacak urutan data training...")
    X_train, y_train = shuffle(X_train, y_train, random_state=42)
    print("✅ Pengacakan selesai!")

except Exception as e:
    print(f"\n❌ ERROR: {e}")

### 2. Membangun Arsitektur Defensive LSTM
Menggunakan **L2(0.001)** dan **Dropout(0.4)** untuk stabilitas tinggi.

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])

model = Sequential([
    Input(shape=input_shape),
    LSTM(32, return_sequences=True, kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    LSTM(16, return_sequences=False, kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

### 3. Pelatihan Model
Patience diatur ke 7 untuk memberi waktu model belajar pola yang halus.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
start_time = time.time()

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_split=0.1,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

print(f"\n✅ Training Selesai dalam {(time.time() - start_time)/60:.2f} menit.")

### 4. Visualisasi & Evaluasi Kinerja
Bagian ini menghasilkan Confusion Matrix dan 5 Metrik Utama untuk pembahasan skripsi.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy Curve')
plt.legend()
plt.show()

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\n" + "="*50)
print("EVALUASI KINERJA: LSTM (DEFENSIVE MODEL)")
print("="*50)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"Confusion Matrix:")
print(f"True Positive  (TP): {tp} (Tebak Naik, Beneran Naik)")
print(f"True Negative  (TN): {tn} (Tebak Turun, Beneran Turun)")
print(f"False Positive (FP): {fp} (Tebak Naik, Eh Ternyata Turun)")
print(f"False Negative (FN): {fn} (Tebak Turun, Eh Ternyata Naik)")

acc = accuracy_score(y_test, y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n5 METRIK UTAMA:")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {pre*100:.2f}%")
print(f"Recall    : {rec*100:.2f}%")
print(f"F1-Score  : {f1*100:.2f}%")
print(f"ROC-AUC   : {auc*100:.2f}%")

print("\nCLASSIFICATION REPORT DETAIL:")
print(classification_report(y_test, y_pred, target_names=['Turun (0)', 'Naik (1)']))
print("="*50)

print("\n--- TARGET BASELINE UNTUK DIBANDINGKAN ---")
print("F1-Score Baseline: 57.05%")
print("ROC-AUC Baseline : 53.34%")